# Collect the 226 brain masks into one archive

Alternative to running `brainiac_probe_features_colab.ipynb` in Colab: this notebook only gathers the
`brainmask.mgz` of the 226 study subjects (about 1.4 GB) from the OASIS-1 FreeSurfer release in Google Drive into
`MyDrive/brainmasks.zip`, so that `scripts/brainiac_features.py` can be run on a local machine
(`--oasis-dir` pointing at the extracted archive). OASIS data remain subject to the OASIS Data Use Agreement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, pandas as pd
OASIS_DIR = '/content/drive/MyDrive/OASIS_Freesurfer'
OUT = '/content/drive/MyDrive/brainmasks.zip'
ids = set(pd.read_csv('https://raw.githubusercontent.com/Yuliawang520/oasis1-cognitive-baseline/main/data/folds.csv')['subject_id'])
found = {}
for root, dirs, files in os.walk(OASIS_DIR):
    if 'brainmask.mgz' in files and os.path.basename(root) == 'mri':
        sid = os.path.basename(os.path.dirname(root))
        if sid in ids:
            found[sid] = os.path.join(root, 'brainmask.mgz')
print(f'{len(found)}/{len(ids)} found; missing: {sorted(ids - set(found))[:10]}')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_STORED) as z:  # .mgz is already gzip-compressed
    for sid, p in sorted(found.items()):
        z.write(p, f'{sid}/mri/brainmask.mgz')
print('written', OUT, round(os.path.getsize(OUT) / 1e9, 2), 'GB')